# 原始特征LightGBM基线

负责人：C。这是中文教学参考，正式实现由成员理解后编写、执行并核对。

输入挂载：官方比赛数据、任务00输出的固定共享Dataset。无需挂载旧track代码包。CPU训练，四线程；机器等待另计。

本项目教学对照：原始特征和基本类别处理。配置为明确的基线设计，不能声称逐项复现某个历史基线成绩。


- [比赛数据与规则](https://www.kaggle.com/competitions/playground-series-s6e9)
- [LightGBM论文](https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html)
- [目标编码：内部交叉拟合与平滑](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)
- [AUC定义](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

本教程生成时尚未执行Kaggle完整训练。历史分数是核对参照，不是本轮结果。阅读当前文档不意味着升级历史环境。

## 如何学习本文件

每次只运行一个单元，先用自己的话预测输出。`iloc`按位置取行，`loc`按标签取行；`to_numpy`去掉索引，之后必须保证位置对应。`assert`是验收条件，失败应查数据而非删除检查。`fit`从数据学习，`transform`使用已学习规则。

编程练习：修改一个小例子的输入并解释变化；正式配置保持历史定义。复杂特征组的整体增益不能归因于单一列。

## 读取官方数据

路径检查避免读错版本；对齐函数先检查ID集合，再恢复官方顺序。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import gc
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

INPUT = Path('/kaggle/input')
TARGET = 'Will_Buy_EV'
SEED = 42
N_SPLITS = 5

def unique_file(name):
    """从已挂载输入中定位唯一文件；多个版本时停止，防止静默读错。"""
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f'Expected one {name}, found {paths}'
    return paths[0]

competition_dirs = [INPUT/'competitions/playground-series-s6e9', INPUT/'playground-series-s6e9']
available = [p for p in competition_dirs if (p/'train.csv').is_file()]
assert len(available) == 1, 'Attach the official competition data.'
DATA = available[0]
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
y = train[TARGET].map({'No':0, 'Yes':1})
assert y.notna().all() and set(y.unique()) == {0,1}
assert train.id.is_unique and test.id.is_unique
assert sample.columns.tolist() == ['id',TARGET] and sample.id.equals(test.id)
assert train.columns.drop(['id',TARGET]).tolist() == test.columns.drop('id').tolist()

def align_rows(frame, ids):
    """先检查一一对应，再按官方顺序排列；不能直接假设CSV行序相同。"""
    assert frame.id.is_unique and len(frame) == len(ids)
    assert set(frame.id) == set(ids)
    return frame.set_index('id').loc[ids].reset_index()

def current_versions():
    return {'lightgbm':lgb.__version__, 'sklearn':sklearn.__version__,
            'numpy':np.__version__, 'pandas':pd.__version__}

## 读取共同分组

共享fold决定训练与验证行，三人不能重新划分。

In [ ]:
fold_path = unique_file('shared_folds.csv')
shared = align_rows(pd.read_csv(fold_path), train.id)
assert np.array_equal(shared.target, y)
assert shared.fold.notna().all() and shared.fold.isin(range(5)).all()
assert set(shared.fold) == set(range(5))
fold_ids = shared.fold.to_numpy(dtype=int)
foundation_note = json.loads((fold_path.parent/'dataset_note.json').read_text())
display(shared.groupby('fold').agg(rows=('id','size'),positive_rate=('target','mean')))
print(current_versions())

## 构造基础特征

只使用原始列。每折从训练部分学习类别词表；未知类别成为缺失值，由LightGBM处理。数值缺失也由模型处理，不用验证数据估计填充值。

In [ ]:
feature_columns = test.columns.drop('id').tolist()
categorical_columns = train[feature_columns].select_dtypes(include=['object','string','category']).columns.tolist()
def prepare_fold(training_index,validation_index):
    """输入两组行位置；输出列一致的训练、验证和测试特征。"""
    frames = [train.iloc[training_index][feature_columns].copy(),train.iloc[validation_index][feature_columns].copy(),test[feature_columns].copy()]
    for column in categorical_columns:
        categories = pd.Index(frames[0][column].astype(str).unique())
        for frame in frames:
            frame[column] = pd.Categorical(frame[column].astype(str),categories=categories)
    return frames
model_params = dict(objective='binary',n_estimators=10000,learning_rate=0.05,num_leaves=31,random_state=42,n_jobs=4,verbosity=-1)
patience, expected_count = 200, None

### 对照上方代码逐句理解

行号从上方代码单元第一行起计。重复操作也列出，方便逐行定位；跨行调用请连同后续参数一起阅读。

| 行 | 代码定位 | 中文解释 |
|---|---|---|
| 7 | `categories = pd.Index(frames[0][column].astype(str).unique())` | 固定数据类型；float32节约内存，int类型表达离散分组，str把数值作为类别键。 |
| 9 | `frame[column] = pd.Categorical(frame[column].astype(str),categories=categories)` | 用训练折词表固定类别编码；未见类别成为缺失，不重新为验证集编号。 |

## 五折训练并回填预测

这是主要耗时单元。每折留出五分之一用于评价；早停也使用这个验证集，因此OOF属于开发评价。predict_proba的[:,1]取购买概率。五次覆盖完成后每行coverage应为1。不要为了整理日志重训。

In [ ]:
run_name = 'baseline'
method_sources = ['https://lightgbm.readthedocs.io/en/stable/Parameters.html']

candidate_oof = np.full(len(train),np.nan)  # 没有预测的行保持NaN，便于发现漏填。
candidate_test = np.zeros(len(test))
coverage = np.zeros(len(train),dtype=np.uint8)
records, fold_features = [], {}
started = perf_counter()
for fold in range(N_SPLITS):
    training_index = np.flatnonzero(fold_ids != fold)
    validation_index = np.flatnonzero(fold_ids == fold)
    X_train, X_valid, X_test = prepare_fold(training_index,validation_index)
    assert X_train.columns.equals(X_valid.columns) and X_train.columns.equals(X_test.columns)
    if expected_count is not None:
        assert X_train.shape[1] == expected_count, (fold,X_train.shape)
        historical_columns = foundation_note.get('historical_features',{}).get(run_name,{}).get(str(fold))
        if historical_columns is not None:
            assert X_train.columns.tolist() == historical_columns, 'Historical feature order differs.'
    fold_features[str(fold)] = X_train.columns.tolist()
    if run_name != 'baseline':
        assert model_params == foundation_note['historical_params'][run_name], 'Historical parameters differ.'
    model = lgb.LGBMClassifier(**model_params)  # 每个外层fold创建全新模型。
    fold_started = perf_counter()
    model.fit(X_train,y.iloc[training_index],eval_set=[(X_valid,y.iloc[validation_index])],
              eval_metric='auc',callbacks=[lgb.early_stopping(patience,first_metric_only=True,verbose=False),lgb.log_evaluation(1000)])
    probability = model.predict_proba(X_valid,num_iteration=model.best_iteration_)[:,1]
    test_probability = model.predict_proba(X_test,num_iteration=model.best_iteration_)[:,1]
    assert np.isfinite(probability).all() and np.isfinite(test_probability).all()
    candidate_oof[validation_index] = probability  # 回填官方训练行的位置。
    coverage[validation_index] += 1
    candidate_test += test_probability/N_SPLITS  # 在概率空间平均，暂不排名。
    records.append({'fold':fold,'auc':float(roc_auc_score(y.iloc[validation_index],probability)),
                    'features':X_train.shape[1],'best_iteration':int(model.best_iteration_),
                    'fit_seconds':perf_counter()-fold_started})
    print(records[-1])
    del model,X_train,X_valid,X_test
    gc.collect()
assert (coverage==1).all() and np.isfinite(candidate_oof).all()
assert ((candidate_oof>=0)&(candidate_oof<=1)).all()
assert ((candidate_test>=0)&(candidate_test<=1)).all()
elapsed = perf_counter()-started
display(pd.DataFrame(records))
print('Overall OOF AUC:',roc_auc_score(y,candidate_oof))

## 保存并交接真实结果

只保存完整OOF、测试预测和摘要。输出目录已经存在时停止，先保存上次结果后重开会话。把整个目录保存为Notebook输出，告诉融合负责人挂载。

In [ ]:
output = Path('/kaggle/working')/run_name
output.mkdir(exist_ok=False)
oof = pd.DataFrame({'id':train.id,'target':y,'fold':fold_ids,'prediction':candidate_oof})
submission = sample.copy()
submission[TARGET] = candidate_test
auc = float(roc_auc_score(y,candidate_oof))
report = (f'The {run_name} model was trained on five shared folds. Overall OOF ROC AUC was {auc:.9f}. '
          'Test probabilities were averaged across five models. All competition-derived supervised '
          'preprocessing was fitted within outer training folds. These are development results; '
          'leaderboard performance for this run remains unverified.')
summary = {'run_name':run_name,'model_params':model_params,'stopping_rounds':patience,
           'fold_features':fold_features,'fold_metrics':records,'oof_auc':auc,'elapsed_seconds':elapsed,
           'versions':current_versions(),'fold_source':str(fold_path),'method_sources':method_sources,
           'test_aggregation':'mean probabilities across five folds','public_score':None,'report_summary':report}
oof.to_csv(output/'oof_predictions.csv',index=False)
submission.to_csv(output/'submission.csv',index=False)
(output/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
print(report)
print('Saved:',output)

## 中文结果解析与理解检查

逐折AUC比较必须使用相同fold。整体OOF AUC和五折AUC的平均不是同一个量。两个完整方案同时改变多组特征，不能宣称某一列造成全部提升。

请回答：为什么测试集不参与早停？为什么目标编码训练行不能直接使用全训练折groupby均值？为什么相同特征数量仍可能有不同列序？请画出一行样本从原始数据到验证预测经过的步骤。

运行后用实际输出写中文观察；摘要已自动生成英文Report Summary。历史参照：收入邻域模型0.946046626，混合特征模型0.946129123；未训练前不能把这些数写成本次结果。